<a href="https://colab.research.google.com/github/mg1206/Analysis-of-SEV-in-ML-Models/blob/main/Sparse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import importlib
from SEV.SEV import SEVPlot, SEV
from SEV import data_loader

from SEV.Encoder import DataEncoder
from sklearn.linear_model import LogisticRegression
import numpy as np # Import numpy

class Args:
    def __init__(self):
        self.dataset = 'german'
args = Args()

#Get the dataset and the negative population
X,y,X_neg = data_loader.data_loader(args.dataset)
#Preprocessing the dataset
encoder = DataEncoder(standard=True)
#Fit the encoder with the negative population
encoder.fit(X_neg)
#Transform the whole dataset
encoded_X = encoder.transform(X)

#Constructing the model
lr = LogisticRegression()
lr.fit(encoded_X,y)

#For explaining the whole dataset, "plus" for SEV+, "minus" for SEV-
SEVPlot(lr,encoder, encoded_X, "plus")

#For explaning the one single instance
sev = SEV(lr,encoder,encoded_X.columns)
#Get the number of sev for this instance
sev_num = sev.sev_cal(np.array(encoded_X.iloc[0]).reshape(1,-1),mode="plus")
print("The SEV Value for instance 0 is %d."%sev_num)
#Get the features can be used in this explanation
features = sev.sev_count(np.array(encoded_X.iloc[0]).reshape(1,-1),mode="plus",choice=sev_num)
print("The feature used in this explanation are %s."%features)

In [ ]:
#Testing stability
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from collections import Counter

# Number of bootstrap models
n_models = 30

# Store results
sev_values = []
feature_choices = []

# Choose instances to evaluate (only positives since SEV+)
model = LogisticRegression(max_iter=1000)
model.fit(encoded_X, y)
preds = model.predict(encoded_X)

positive_indices = np.where(preds == 1)[0]

# Evaluate first 50 positive instances
eval_indices = positive_indices[:50]

print(f"Evaluating {len(eval_indices)} positive instances")

# Bootstrap loop
for i in range(n_models):
    print(f"Training model {i+1}/{n_models}")

    # Bootstrap sample
    X_boot, y_boot = resample(encoded_X, y, replace=True)

    # Train logistic regression
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_boot, y_boot)

    # Initialize SEV
    sev = SEV(lr, encoder, encoded_X.columns)

    model_sev_vals = []
    model_features = []

    for idx in eval_indices:
        x_instance = np.array(encoded_X.iloc[idx]).reshape(1, -1)

        # Compute SEV value
        sev_num = sev.sev_cal(x_instance, mode="plus")
        model_sev_vals.append(sev_num)

        # Get feature set
        features = sev.sev_count(
            x_instance,
            mode="plus",
            choice=sev_num
        )

        model_features.append(tuple(sorted(features)))

    sev_values.append(model_sev_vals)
    feature_choices.append(model_features)

print("Done training all models.")

In [ ]:
sev_values = np.array(sev_values)

# SEV stability
sev_std = np.std(sev_values, axis=0)
print("Average SEV standard deviation across instances:", np.mean(sev_std))

# Feature agreement
agreement_rates = []

for instance_idx in range(len(eval_indices)):
    features_across_models = [
        feature_choices[m][instance_idx]
        for m in range(n_models)
    ]

    counts = Counter(features_across_models)
    most_common_count = counts.most_common(1)[0][1]

    agreement = most_common_count / n_models
    agreement_rates.append(agreement)

print("Average feature agreement rate:", np.mean(agreement_rates))